# 02. Línea base

Una CNN sencilla entrenada desde cero con la configuración de `configs/base.toml`. Este notebook no busca un buen número. Busca ver cómo se comporta la red para decidir qué vale la pena cambiar.

Todo el trabajo lo hace el paquete `nubes`. El notebook y la línea de comandos llaman a las mismas funciones. Test no aparece aquí.

## 0. Parámetros

`SEEDS` son las semillas de entrenamiento. El split no cambia con ellas. Una corrida que ya existe en `runs/` se carga en vez de repetirse.

In [ ]:
SEEDS = [11, 22, 33]

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image

from nubes import eda, figuras
from nubes.config import RunConfig
from nubes.datos import image_root, load_manifest, make_dataset
from nubes.experimento import build_model, load_model, run_seeds
from nubes.inspeccion import feature_maps, receptive_field, shape_trace
from nubes.metricas import group_confusion, symmetric_confusions, top_confusions, within_group_share
from nubes.modelo import count_parameters
from nubes.resumen import aggregate, runs_table, total_confusion
from nubes.utilidades import PROJECT_ROOT, pick_device

CONFIGS = PROJECT_ROOT / "configs"
base = RunConfig.from_toml(CONFIGS / "base.toml")
manifest, stats = load_manifest(base.data_dir)
ROOT = image_root(stats)
CLASSES = stats["classes"]
device = pick_device(base.device)
print(device, "|", base.epochs, "épocas |", "semillas", SEEDS)

## 1. Arquitectura y campo receptivo

Cada bloque es `Conv 3x3 → ReLU → MaxPool 2x2`. La tabla de campo receptivo dice qué lado de la imagen de entrada ve una posición del mapa de salida de cada bloque. `fraction` es ese lado como fracción de la imagen.

Después del último bloque la red promedia todo el mapa. Con `head_grid = 1` la posición dentro de la imagen se pierde.

In [ ]:
model = build_model(base, len(CLASSES))
print(f"{count_parameters(model):,} parámetros")
display(shape_trace(model, base.image_size))
receptive_field(model, base.image_size)

## 2. Piloto

Una sola corrida de la línea base con más épocas de las necesarias. Sirve para ver dónde está el mínimo de `val_loss` y fijar un número de épocas común para todos los experimentos.

La decisión se escribe en `docs/PROTOCOLO.md` y se pone en `epochs` de `configs/base.toml` antes de seguir.

In [ ]:
pilot = RunConfig.from_toml(CONFIGS / "piloto.toml")
pilot_result = run_seeds(pilot, [SEEDS[0]])[0]
figuras.plot_history(pilot_result.history, f"Piloto. Mejor época: {pilot_result.best_epoch}");

In [ ]:
history = pilot_result.history.set_index("epoch")
marks = sorted({pilot_result.best_epoch, *range(10, pilot.epochs + 1, 10)})
history.loc[marks, ["train_loss", "train_acc", "val_loss", "val_acc", "val_macro_f1"]].round(4)

**Observación:**

*por escribir*

## 3. Línea base con varias semillas

Misma configuración y mismo split con tres semillas distintas. La dispersión entre semillas es la vara para medir cualquier diferencia entre configuraciones más adelante.

In [ ]:
results = run_seeds(base, SEEDS)
table = runs_table({base.name: results})
display(table.round(4))
aggregate(table).round(4)

In [ ]:
for result in results:
    figuras.plot_history(
        result.history, f"{base.name} seed {result.config.seed}. Mejor época: {result.best_epoch}"
    )

In [ ]:
n_val = sum(results[0].val.support)
spread = table["val_acc"].max() - table["val_acc"].min()
print("imágenes de validation:", n_val)
print("rango de val_acc entre semillas:", round(spread, 4), "=", round(spread * n_val), "imágenes")
print("gap medio train - val:", round(table["gap"].mean(), 4))

**Observación:**

*por escribir*

## 4. Qué clases se confunden

La matriz suma las tres semillas. El color está normalizado por fila y el número es el conteo.

In [ ]:
confusion = total_confusion(results)
figuras.plot_matrix(
    confusion, CLASSES, normalize=True, title=f"{base.name}. Validation, {len(SEEDS)} semillas"
);

In [ ]:
per_class = pd.concat([r.val.per_class(CLASSES) for r in results]).groupby(level=0).mean()
display(per_class.loc[CLASSES].round(3))
display(top_confusions(confusion, CLASSES, k=10))
symmetric_confusions(confusion, CLASSES, k=10)

Los errores también se pueden leer por familia. `dentro` es la fracción de los errores que cae entre clases del mismo grupo. `azar` es la fracción de pares de clases distintas que comparten grupo.

In [ ]:
info = eda.class_table().loc[CLASSES]
off_diagonal = ~np.eye(len(CLASSES), dtype=bool)
rows = {}
for key in ("level", "form"):
    groups = info[key].to_numpy()
    rows[key] = {
        "dentro": within_group_share(confusion, list(groups)),
        "azar": (groups[:, None] == groups[None])[off_diagonal].mean(),
    }
    display(group_confusion(confusion, list(groups)))
pd.DataFrame(rows).T.round(3)

**Observación:**

*por escribir*

## 5. Las confusiones de la CNN frente a las del EDA

Se repite el k vecinos del notebook 01 y se compara su matriz de confusión con la de la CNN. La correlación de Spearman se calcula sobre los 55 pares de clases con los errores sumados en ambos sentidos.

Una correlación alta dice que la red se equivoca donde las clases ya se parecían en rasgos globales.

In [ ]:
thumbs = eda.load_thumbnails(manifest, ROOT, size=32)
_, cm_knn = eda.knn_baseline(eda.simple_features(thumbs).to_numpy(), manifest)
upper = np.triu_indices(len(CLASSES), k=1)


def pair_errors(matrix) -> pd.Series:
    values = np.asarray(matrix)
    names = [f"{CLASSES[i]} / {CLASSES[j]}" for i, j in zip(*upper, strict=True)]
    return pd.Series((values + values.T)[upper], index=names)


both = pd.DataFrame({"cnn": pair_errors(confusion), "knn": pair_errors(cm_knn)})
print("Spearman:", round(both["cnn"].corr(both["knn"], method="spearman"), 3))
both.sort_values("cnn", ascending=False).head(12)

**Observación:**

*por escribir*

## 6. Las imágenes que fallan

`predictions_val.csv` guarda la predicción de cada imagen de validation. Primero se cuenta en cuántas semillas falla cada imagen. Las que fallan en todas son difíciles para esta arquitectura. Las que fallan en una sola son ruido de entrenamiento.

In [ ]:
predictions = pd.concat(
    pd.read_csv(r.run_dir / "predictions_val.csv").assign(seed=r.config.seed) for r in results
)
predictions["error"] = predictions["real"] != predictions["predicha"]
failures = predictions.groupby("path")["error"].sum()
failures.value_counts().sort_index().rename("imágenes").rename_axis(
    "semillas que fallan"
).to_frame()

In [ ]:
always = failures[failures == len(SEEDS)].index
hard = predictions[predictions["path"].isin(always) & (predictions["seed"] == SEEDS[0])]
hard = hard.sort_values("confianza", ascending=False).head(18)
if len(hard):
    figuras.plot_image_grid(
        [Image.open(ROOT / path).convert("RGB") for path in hard["path"]],
        [f"{r.real}→{r.predicha} {r.confianza:.2f}" for r in hard.itertuples()],
        ncols=6,
        cell=2.1,
    )
hard["real"].value_counts().rename("fallan siempre").to_frame().T

**Observación:**

*por escribir*

## 7. Feature maps después de entrenar

Salida de la ReLU del primer bloque y del último para una imagen de validation.

In [ ]:
trained, trained_config, _ = load_model(results[0].run_dir, device)
val_dataset = make_dataset(trained_config, "val")
x, y = val_dataset[0]
figuras.plot_image_grid(
    list(figuras.denormalize(x.unsqueeze(0), stats["mean_train"], stats["std_train"])),
    [CLASSES[y]],
    ncols=1,
    cell=2.6,
)
for layer in ("stages.s1.act", f"stages.s{trained_config.blocks}.act"):
    maps = feature_maps(trained, x.unsqueeze(0), layer)[0]
    figuras.plot_feature_maps(maps, f"{layer} {tuple(maps.shape)}")

## 8. Cierre

De aquí sale la hipótesis. En `docs/BITACORA.md` queda escrito:

- qué mostró la línea base y con qué evidencia
- la pregunta del proyecto
- la hipótesis, lo que se espera ver y por qué
- qué variable se va a cambiar y cuáles quedan fijas